  # Whisper Accent Robustness — Model Performance Evaluation







  Run `eval_model_perf.py` on SLURM first to generate the CSVs this notebook reads.







  - **Combined** → scripted + spontaneous test data (single CSV per model)

  - **Per-domain** → filter by `domain` column within each CSV







  Metrics:



  - **WER**  — word error rate (primary ASR metric)

  - **PER**  — phoneme error rate via G2P; labelled "PER (G2P)" throughout

In [22]:
# ── Config ────────────────────────────────────────────────────────────────────
RESULTS_DIR = "results/model_perf_comparison"


# Keys must match {model_key} in CSV filenames; values are display labels
MODEL_KEYS = {
    "baseline": "Zero-shot",
    "whisper_ft": "Naive LoRA FT",
    "whisper_ft_hoc": "Naive LoRA FT (heldout Chinese)",
    "whisfusion": "WhisFusion (Zero-shot)",
    "whisfusion_ft": "WhisFusion (LoRA FT)",
    "whisfusion_ft_hoc": "WhisFusion (LoRA FT, heldout Chinese)",
}
DOMAINS = ["scripted", "spontaneous"]  # Filter options within combined CSVs




In [23]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
import jiwer

pd.set_option("display.max_colwidth", 80)




In [24]:
def load_results(model_key: str) -> pd.DataFrame | None:
    """Load combined predictions CSV (scripted + spontaneous)."""
    p = Path(RESULTS_DIR) / f"{model_key}_predictions.csv"
    if not p.exists():
        print(f"  [missing] {p} — run eval_model_perf.py first")
        return None
    return pd.read_csv(p)


def filter_by_domain(df: pd.DataFrame, domain: str | None) -> pd.DataFrame:
    """Filter combined DataFrame by domain (or return all)."""
    if domain is None:
        return df
    return df[df["domain"] == domain].copy()


# ── Load all cached CSVs ──────────────────────────────────────────────────────
results: dict[str, pd.DataFrame | None] = {}
for key in MODEL_KEYS:
    df = load_results(key)
    results[key] = df
    if df is not None:
        print(f"  loaded  {key}: {len(df):,} rows ({df['domain'].value_counts().to_dict()})")
    else:
        print(f"  [missing] {key}")



  loaded  baseline: 8,237 rows ({'scripted': 7796, 'spontaneous': 441})
  loaded  whisper_ft: 8,237 rows ({'scripted': 7796, 'spontaneous': 441})
  loaded  whisper_ft_hoc: 8,237 rows ({'scripted': 7796, 'spontaneous': 441})
  loaded  whisfusion: 8,237 rows ({'scripted': 7796, 'spontaneous': 441})
  loaded  whisfusion_ft: 8,237 rows ({'scripted': 7796, 'spontaneous': 441})
  [missing] results/model_perf_comparison/whisfusion_ft_hoc_predictions.csv — run eval_model_perf.py first
  [missing] whisfusion_ft_hoc


In [25]:
# ── Helpers ──────────────────────────────────────────────────────────────────


def available(key: str) -> bool:
    return results.get(key) is not None


def get_data(key: str, domain: str | None = None) -> pd.DataFrame | None:
    """Get data for model, optionally filtered by domain."""
    df = results.get(key)
    if df is None:
        return None
    return filter_by_domain(df, domain)


def corpus_wer(df: pd.DataFrame) -> float:
    return float(jiwer.wer(
        df["reference_norm"].fillna("").tolist(),
        df["prediction_norm"].fillna("").tolist(),
    ))

def corpus_mer(df: pd.DataFrame) -> float:
    return float(jiwer.mer(
        df["reference_norm"].fillna("").tolist(),
        df["prediction_norm"].fillna("").tolist(),
    ))

def corpus_per(df: pd.DataFrame) -> float | None:
    """Mean utterance PER (G2P-derived)."""
    if "utt_per" not in df.columns:
        return None
    vals = df["utt_per"].dropna()
    return float(vals.mean()) if len(vals) else None


def grouped_wer(df: pd.DataFrame, group_col: str = "l1") -> pd.DataFrame:
    rows = []
    for grp, sub in df.groupby(group_col):
        rows.append({
            group_col:  grp,
            "num_utts": len(sub),
            "wer":      float(jiwer.wer(
                                sub["reference_norm"].fillna("").tolist(),
                                sub["prediction_norm"].fillna("").tolist(),
                            )),
            "mer":      float(jiwer.mer(
                                sub["reference_norm"].fillna("").tolist(),
                                sub["prediction_norm"].fillna("").tolist(),
                            )),
            "per":      float(sub["utt_per"].dropna().mean())
                        if "utt_per" in sub.columns else None,
        })
    return pd.DataFrame(rows)


print("Helpers loaded.")



Helpers loaded.


  ---



  # Part 1 — Overall WER & PER (G2P)



  **Default: Combined (all data).** Use `domain="scripted"` or `"spontaneous"` in `get_summary_table()` for domain-specific views.

In [26]:
def get_summary_table(domain: str | None = None) -> pd.DataFrame:
    """WER/PER table for domain (None = all data)."""
    rows = []
    for key, label in MODEL_KEYS.items():
        df = get_data(key, domain)
        if df is None:
            continue
        wer = corpus_wer(df)
        per = corpus_per(df)
        mer = corpus_mer(df)
        rows.append({"Model": f"{label}", "WER": wer, "MER": mer, "PER (G2P)": per})
    return pd.DataFrame(rows)


domain = None
df = get_summary_table(domain)
display(
    df.style
    .format({
        "WER": "{:.4f}",
        "MER": "{:.4f}",
        "PER (G2P)": lambda v: f"{v:.4f}" if pd.notna(v) else "—",
    })
    .background_gradient(cmap="RdYlGn_r", subset=["WER", "MER", "PER (G2P)"])
    .set_caption("WER & PER (G2P) — All Data (Scripted + Spontaneous)")
)


,Model,WER,MER,PER (G2P)
0,Zero-shot,0.1546,0.1525,0.0826
1,Naive LoRA FT,0.0827,0.0816,0.0539
2,Naive LoRA FT (heldout Chinese),0.0852,0.0842,0.0512
3,WhisFusion (Zero-shot),0.2758,0.2686,0.1519
4,WhisFusion (LoRA FT),0.1740,0.1714,0.1027


In [32]:
from plotly.subplots import make_subplots


def plot_summary_bars(domain: str | None = None):
    """Plot WER/PER bars for domain (None = all data)."""
    sub = get_summary_table(domain)
    if sub.empty:
        return

    domain_str = f" ({domain})" if domain else " (All Data)"

    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=("WER", "MER", "PER (G2P)"),
        vertical_spacing=0.12,
    )

    x = sub["Model"].tolist()

    for row, col_name, label in [
        (1, "WER", "WER"),
        (2, "MER", "MER"),
        (3, "PER (G2P)", "PER (G2P)"),
    ]:
        vals = sub[col_name]
        if col_name == "PER (G2P)" and not vals.notna().any():
            continue

        fig.add_trace(
            go.Bar(
                name=label,
                x=x,
                y=vals.tolist(),
                text=[f"{v:.1%}" if pd.notna(v) else "" for v in vals],
                textposition="outside",
                showlegend=False,
                cliponaxis=False,
            ),
            row=row, col=1
        )

    fig.update_yaxes(title_text="Error Rate", tickformat=".0%", row=1, col=1)
    fig.update_yaxes(tickformat=".0%", row=2, col=1)
    fig.update_yaxes(tickformat=".0%", row=3, col=1)

    fig.update_xaxes(tickangle=-20, automargin=True)

    fig.update_layout(
        title=f"WER vs PER by Model{domain_str}",
        height=950,
        width=1200,
        margin=dict(t=100, b=80, l=60, r=40),
        bargap=0.35,
    )
    fig.show()

domain = "scripted"
plot_summary_bars(domain)


  ---



  # Part 2 — Per-L1 WER



  **Default: Combined (all data).** Use `domain=` for domain-specific L1 breakdowns.

In [33]:
def get_l1_breakdown(domain: str | None = None, metric: str = "wer") -> pd.DataFrame:
    """Per-L1 breakdown for a chosen metric."""
    metric = metric.lower()
    if metric not in {"wer", "mer", "per"}:
        raise ValueError("metric must be one of: 'wer', 'mer', 'per'")

    l1_rows = []
    for key, label in MODEL_KEYS.items():
        df = get_data(key, domain)
        if df is None:
            continue
        grp = grouped_wer(df, "l1")
        grp["Model"] = label
        l1_rows.append(grp)

    if not l1_rows:
        return pd.DataFrame()

    l1_df = pd.concat(l1_rows, ignore_index=True)

    # Delta vs zero-shot baseline (negative = improvement)
    if "baseline" in MODEL_KEYS and available("baseline"):
        base_label = MODEL_KEYS["baseline"]
        base_col = f"{metric}_base"
        base_vals = (
            l1_df[l1_df["Model"] == base_label][["l1", metric]]
            .rename(columns={metric: base_col})
        )
        l1_df = l1_df.merge(base_vals, on="l1", how="left")
        l1_df[f"{metric}_delta_pct"] = (
            (l1_df[metric] - l1_df[base_col]) / l1_df[base_col] * 100
        )

    return l1_df


domain = "scripted"   # None / "scripted" / "spontaneous"
metric = "mer"  # "wer" / "mer" / "per"

l1_df = get_l1_breakdown(domain, metric)

if not l1_df.empty:
    l1s = sorted(l1_df["l1"].unique())
    metric_label = "PER (G2P)" if metric == "per" else metric.upper()

    # Use actual model keys that exist in MODEL_KEYS and are available
    model_order = list(MODEL_KEYS.keys())
    ordered_items = [(k, MODEL_KEYS[k]) for k in model_order if available(k)]

    fig = go.Figure()
    for key, label in ordered_items:
        sub = l1_df[l1_df["Model"] == label].set_index("l1")
        fig.add_trace(go.Bar(
            name=label,
            x=l1s,
            y=[sub.loc[l, metric] if l in sub.index else None for l in l1s],
            text=[f"{sub.loc[l, metric]:.1%}" if l in sub.index and pd.notna(sub.loc[l, metric]) else "" for l in l1s],
            textposition="outside",
        ))

    domain_str = f" (domain={domain})" if domain else " (All Data)"
    fig.update_layout(
        title=f"{metric_label} by L1{domain_str}",
        barmode="group",
        yaxis=dict(title=metric_label, tickformat=".0%"),
        legend=dict(orientation="h", y=1.12, xanchor="center", x=0.5),
        margin=dict(t=80),
    )
    fig.show()

    out = Path(RESULTS_DIR) / f"comparison_{domain or 'all'}_{metric}_by_l1.csv"
    l1_df.to_csv(out, index=False)
    print(f"Saved {out}")

    display(
        l1_df.sort_values(["l1", "Model"])
            .style.format({
                metric: "{:.4f}",
                f"{metric}_base": "{:.4f}",
                f"{metric}_delta_pct": "{:+.1f}%",
                "per": lambda v: f"{v:.4f}" if pd.notna(v) else "—",
            })
            .set_caption(f"{domain_str} — Per-L1 {metric_label}")
    )


Saved results/model_perf_comparison/comparison_scripted_mer_by_l1.csv


,l1,num_utts,wer,mer,per,Model,mer_base,mer_delta_pct
7,Arabic,1132,0.040262,0.0400,0.0228,Naive LoRA FT,0.1252,-68.0%
14,Arabic,1132,0.043643,0.0434,0.0244,Naive LoRA FT (heldout Chinese),0.1252,-65.3%
28,Arabic,1132,0.066607,0.0659,0.0502,WhisFusion (LoRA FT),0.1252,-47.4%
21,Arabic,1132,0.245452,0.2392,0.1269,WhisFusion (Zero-shot),0.1252,+91.1%
0,Arabic,1132,0.126752,0.1252,0.0578,Zero-shot,0.1252,+0.0%
8,Chinese,1130,0.088053,0.0873,0.0546,Naive LoRA FT,0.1886,-53.7%
15,Chinese,1130,0.108696,0.1073,0.0669,Naive LoRA FT (heldout Chinese),0.1886,-43.1%
29,Chinese,1130,0.107898,0.1055,0.0823,WhisFusion (LoRA FT),0.1886,-44.1%
22,Chinese,1130,0.319904,0.3104,0.1856,WhisFusion (Zero-shot),0.1886,+64.6%
1,Chinese,1130,0.191564,0.1886,0.1060,Zero-shot,0.1886,+0.0%


  ---



  # Part 3 — Domain Gap Analysis (zero-shot)



  Scripted vs spontaneous **within** each L1, using baseline model.

In [29]:
def plot_domain_gap(model_key: str = "baseline"):
    """Scripted vs spontaneous gap by L1."""
    df = results.get(model_key)
    if df is None:
        return
    
    s_g  = grouped_wer(df[df["domain"] == "scripted"], "l1").rename(columns={"wer": "WER_scripted"})
    sp_g = grouped_wer(df[df["domain"] == "spontaneous"], "l1").rename(columns={"wer": "WER_spontaneous"})
    
    gap = s_g[["l1", "WER_scripted"]].merge(
        sp_g[["l1", "WER_spontaneous"]], on="l1", how="inner")
    gap["gap"] = gap["WER_spontaneous"] - gap["WER_scripted"]
    
    if gap.empty:
        print(f"No domain gap data for {model_key}")
        return
    
    l1s = gap["l1"].tolist()
    fig = go.Figure()
    fig.add_trace(go.Bar(name="Scripted", x=l1s, y=gap["WER_scripted"].tolist()))
    fig.add_trace(go.Bar(name="Spontaneous", x=l1s, y=gap["WER_spontaneous"].tolist()))
    fig.update_layout(
        title=f"{MODEL_KEYS.get(model_key, model_key)} — Scripted vs Spontaneous by L1",
        barmode="group",
        yaxis=dict(title="WER", tickformat=".0%"),
        legend=dict(orientation="h", y=1.12, xanchor="center", x=0.5),
    )
    fig.show()
    display(
        gap.style
         .format({c: "{:.4f}" for c in ["WER_scripted", "WER_spontaneous", "gap"]})
         .set_caption(f"{MODEL_KEYS.get(model_key, model_key)} — Domain Gap by L1")
    )


plot_domain_gap("baseline")



,l1,WER_scripted,WER_spontaneous,gap
0,Arabic,0.1268,0.1705,0.0437
1,Chinese,0.1916,0.0942,-0.0974
2,English,0.0370,0.1688,0.1318
3,Hindi,0.0653,0.0645,-0.0008
4,Korean,0.0805,0.1364,0.0559
5,Spanish,0.2284,0.2296,0.0013
6,Vietnamese,0.3117,0.1111,-0.2006


  ---



  # Part 4 — Utterance-level Analysis



  **Default: Combined**. Filter by `domain=` for specific views.

In [30]:
# ── Utterance-level metric distributions ──────────────────────────────────────
def plot_utt_metric_dist(
    domain: str | None = None,
    metric: str = "wer",      # "wer" / "mer" / "per"
    plot_kind: str = "cdf",   # "cdf" / "hist"
):
    metric = metric.lower()
    plot_kind = plot_kind.lower()

    if domain is not None and domain not in DOMAINS:
        raise ValueError(f"domain must be one of {DOMAINS} or None")
    if metric not in {"wer", "mer", "per"}:
        raise ValueError("metric must be one of: 'wer', 'mer', 'per'")
    if plot_kind not in {"cdf", "hist"}:
        raise ValueError("plot_kind must be 'cdf' or 'hist'")

    metric_map = {
        "wer": ("utt_wer", "WER"),
        "mer": ("utt_mer", "MER"),
        "per": ("utt_per", "PER (G2P)"),
    }
    metric_col, metric_label = metric_map[metric]

    fig = go.Figure()
    n_traces = 0

    for key, label in MODEL_KEYS.items():
        df = get_data(key, domain)
        if df is None or metric_col not in df.columns:
            continue

        vals = pd.to_numeric(df[metric_col], errors="coerce").dropna().to_numpy()
        if len(vals) == 0:
            continue

        if plot_kind == "cdf":
            vals = np.sort(vals)
            y = np.arange(1, len(vals) + 1) / len(vals)
            fig.add_trace(go.Scatter(x=vals, y=y, mode="lines", name=label, line=dict(width=2)))
        else:
            fig.add_trace(
                go.Histogram(
                    x=vals,
                    name=label,
                    opacity=0.35,
                    histnorm="probability density",
                    nbinsx=40,
                )
            )
        n_traces += 1

    if n_traces == 0:
        print(f"No data available for metric='{metric}' and domain='{domain}'.")
        return

    domain_str = f" ({domain})" if domain else " (All Data)"
    fig.update_layout(
        title=f"{metric_label} {('Cumulative Distribution' if plot_kind == 'cdf' else 'Distribution')}{domain_str}",
        xaxis=dict(title=metric_label, tickformat=".0%"),
        yaxis=dict(
            title="Cumulative probability" if plot_kind == "cdf" else "Density",
            tickformat=".0%" if plot_kind == "cdf" else None,
        ),
        legend=dict(orientation="h", y=1.12, xanchor="center", x=0.5),
        template="plotly_white",
        margin=dict(t=90),
        barmode="overlay" if plot_kind == "hist" else None,
    )
    fig.show()


domain = None
metric = "mer"
plot_kind = "hist"
plot_utt_metric_dist(domain=domain, metric=metric, plot_kind=plot_kind)


  ---



  # Part 5 — Worst Utterances (cross-model)



  **Default: Combined**. Use `domain=` for filtering.

In [31]:
# ── Worst utterances per model — cross-model comparison ───────────────────────
N_WORST = 30
models = [
    "baseline",
    "whisper_ft",
    "whisfusion_ft",
]
domain = "spontaneous"  # None = all data; "scripted"; "spontaneous"
exclude_l1s = ["Jamaican", "English"]  # e.g. ["Jamaican"]; use [] to keep all L1s
metric = "mer"  # "wer" / "mer" / "per"

if metric not in {"wer", "mer", "per"}:
    raise ValueError("metric must be one of: 'wer', 'mer', 'per'")

metric_col = f"utt_{metric}"

base_key = models[0]
base_df = get_data(base_key, domain)
if base_df is None:
    raise ValueError(f"Missing data for {base_key}")

for anchor_key in models:
    if not available(anchor_key):
        continue

    anchor_df = get_data(anchor_key, domain)
    if anchor_df is None:
        continue

    if exclude_l1s:
        anchor_df = anchor_df[~anchor_df["l1"].isin(exclude_l1s)].copy()
        base_df_f = base_df[~base_df["l1"].isin(exclude_l1s)].copy()
    else:
        base_df_f = base_df

    if metric_col not in anchor_df.columns:
        print(f"Skipping {anchor_key}: missing {metric_col}")
        continue

    worst_ids = anchor_df.nlargest(N_WORST, metric_col)["utterance_id"].tolist()

    worst = base_df_f.set_index("utterance_id").loc[
        worst_ids, ["speaker", "l1", "domain", "reference_norm"]
    ].reset_index()

    for key in models:
        if not available(key):
            continue
        other = get_data(key, domain)
        if exclude_l1s:
            other = other[~other["l1"].isin(exclude_l1s)].copy()

        other = other.set_index("utterance_id").loc[worst_ids]
        worst[f"pred_{key}"] = other["prediction_norm"].values
        worst[f"{metric}_{key}"] = other[metric_col].values

    domain_str = f" (domain={domain})" if domain else " (All Data)"
    exclude_str = f" | excluded L1s: {', '.join(exclude_l1s)}" if exclude_l1s else ""
    metric_label = metric.upper()
    fmt_cols = {c: "{:.3f}" for c in worst.columns if c.startswith(f"{metric}_")}

    display(
        worst.style
        .format(fmt_cols)
        .set_properties(**{
            "max-width": "300px",
            "white-space": "normal",
            "overflow-wrap": "break-word",
            "word-break": "break-word",
            "overflow": "visible",
        })
        .set_caption(
            f"Top-{N_WORST} Worst Utterances by {metric_label} for {MODEL_KEYS[anchor_key]}{domain_str}{exclude_str}"
        )
    )


,utterance_id,speaker,l1,domain,reference_norm,pred_baseline,mer_baseline,pred_whisper_ft,mer_whisper_ft,pred_whisfusion_ft,mer_whisfusion_ft
0,ebvs_chunk3,EBVS,Spanish,spontaneous,the the real their real suitcase and want and the man look at one red dress and the woman uh yellow tie and it happening,the real sweet case and the men look at one red dress and the woman a yellow tee and it happened in,0.423,the their rare their rare sweet case and one and the man looked at one red dress and the woman a yellow tee and it happened,0.385,there their uh case and the and the man looks at a suitcase and suit suit suit suit suit suit suit suit suit suit suit suit,0.786
1,ebvs_chunk4,EBVS,Spanish,spontaneous,when they when the people are really go go from s from somewhere and they really it doesnt matter because,when the people are really go from somewhere and they really it doesnt matter because,0.250,when the when the people are really go go from so from somewhere and they really it doesnt matter because,0.100,when when the they are uhhhhhhhh uh u u where,0.800
2,bwc_chunk2,BWC,Chinese,spontaneous,and they just falled down and since their cases are the same they didnt pay attention to that so they just grabbed one and go to their go uh go to their destinations,and they just fold it down and since their cases are the same they didnt pay attention to that so they just grab one and go to their destinations,0.235,and they just falled down and since their cases are the same they didnt pay attention to that so they just grab one and go to their go go to their destinations,0.061,and just justied up and they are to their to to their they they they,0.788
3,zhaa_chunk0,ZHAA,Arabic,spontaneous,k well the first photo show shows a uh high building it seems to be a modern busy city and then the second one shows a male and a,okay well the first photo shows a high building it seems to be a modern business city and then this second one shows a email and a,0.207,ok well the first photo shows a high building it seems to be a modern presidency and then the second one shows an email and a,0.241,ok the the first firsth is is a highh and and ishh digyy,0.862
4,ebvs_chunk0,EBVS,Spanish,spontaneous,this is a history like it happened in a big city is probably new york or something like that there are two kinds of of house one a big apartments or buildings for business um,this is a history like it happened in a big city probably in new york or something like that there are two kinds of houses one a big apartment or building for business,0.200,this is a history like it happened in a big city miss probably new york or something like that there are two kinds of a house one a big apartments our buildings for business um,0.086,this is a history it happened in a big city is a a it it is like one of one of her one,0.600
5,hjk_chunk1,HJK,Korean,spontaneous,was late after he finished his uh business he was supposed to go to the airport early in the morning but he was late for the flight so while rushing to,he was late after he finished his business he was supposed to go to the airport early in the morning but he was late for the flight,0.188,was late after he finished his business he was supposed to go to the airport early in the morning but he was late for the flight so while rushing to,0.032,was after after he finished his his his was to to to the and and the morning and the morningish the,0.613
6,zhaa_chunk1,ZHAA,Arabic,spontaneous,female then the male is wearing um a suit and the female is wearing a skirt and shes also wearing a suit uh both holding uh suitcases uh they,female the male is wearing a suit and the female is wearing a skirt and shes also wearing a suit both holding suitcases they,0.172,female then the male is wearing uh suit and the female is wearing a skirt and shes also wearing a suit uh both holding uh suitcases uh they,0.069,woman the the man is carryinging and and and woman woman and and and and also also also also also woman also a woman and a woma

,utterance_id,speaker,l1,domain,reference_norm,pred_baseline,mer_baseline,pred_whisper_ft,mer_whisper_ft,pred_whisfusion_ft,mer_whisfusion_ft
0,ebvs_chunk3,EBVS,Spanish,spontaneous,the the real their real suitcase and want and the man look at one red dress and the woman uh yellow tie and it happening,the real sweet case and the men look at one red dress and the woman a yellow tee and it happened in,0.423,the their rare their rare sweet case and one and the man looked at one red dress and the woman a yellow tee and it happened,0.385,there their uh case and the and the man looks at a suitcase and suit suit suit suit suit suit suit suit suit suit suit suit,0.786
1,zhaa_chunk0,ZHAA,Arabic,spontaneous,k well the first photo show shows a uh high building it seems to be a modern busy city and then the second one shows a male and a,okay well the first photo shows a high building it seems to be a modern business city and then this second one shows a email and a,0.207,ok well the first photo shows a high building it seems to be a modern presidency and then the second one shows an email and a,0.241,ok the the first firsth is is a highh and and ishh digyy,0.862
2,ebvs_chunk2,EBVS,Spanish,spontaneous,instead they crash each other and after that ø they get home they figure out like their suitcase like get everyone it not,instead they crash each other and after that when they get home they figure out like their seatquakes like get everyone is not,0.130,instead they crash each other and after that when they get home they figure out like their seat grays like get everyone is not,0.167,instead they they crashed each each they they they they they not not not not,0.750
3,ebvs_chunk4,EBVS,Spanish,spontaneous,when they when the people are really go go from s from somewhere and they really it doesnt matter because,when the people are really go from somewhere and they really it doesnt matter because,0.250,when the when the people are really go go from so from somewhere and they really it doesnt matter because,0.100,when when the they are uhhhhhhhh uh u u where,0.800
4,hjk_chunk4,HJK,Korean,spontaneous,he took the wrong one the wrong suitcase because there was a womans inner wear in the suitcase and then thats the same woman after got she got back to the hotel she,he took the wrong one the wrong suitcase because there was a womans in a rare in the suitcase and then thats the same woman after she got back to the hotel she,0.118,he took the wrong one the wrong suitcase because there was a womans underwear in the suitcase and then the same woman after got she got back to the hotel she,0.091,he took the one one same suitcase was the woman in the samecasecase and the the the the the the,0.667
5,svbi_chunk1,SVBI,Hindi,spontaneous,fall down fall next to each other and take their own respective luggages and start walking they reach home only to realize that they actually took the other persons luggage the man starts seeing,fall down apologize to each other and take their own respective luggage and start walking they reach home only to realize that they actually took the other persons luggage the man starts seeing,0.088,fall down apologize to each other and take their own respective luggage and start walking they reach home only to realize that they actually took the other persons luggage the man starts seeing,0.088,fall long and apologize to to each and take their poage their start to start their start and they had the the the the the the,0.714
6,ebvs_chunk0,EBVS,Spanish,spontaneous,this is a history like it happened in a big city is probably new york or something like that there are two kinds of of house one a big apartments or buildings for business um,this is a history like it happened in a big city probably in new york or something like that there are two kinds of houses one a big apartment or building for business,0.200,this is a history like it happened in a big city miss probably new york or something like that there are two kinds of a house one a big 

,utterance_id,speaker,l1,domain,reference_norm,pred_baseline,mer_baseline,pred_whisper_ft,mer_whisper_ft,pred_whisfusion_ft,mer_whisfusion_ft
0,zhaa_chunk0,ZHAA,Arabic,spontaneous,k well the first photo show shows a uh high building it seems to be a modern busy city and then the second one shows a male and a,okay well the first photo shows a high building it seems to be a modern business city and then this second one shows a email and a,0.207,ok well the first photo shows a high building it seems to be a modern presidency and then the second one shows an email and a,0.241,ok the the first firsth is is a highh and and ishh digyy,0.862
1,zhaa_chunk1,ZHAA,Arabic,spontaneous,female then the male is wearing um a suit and the female is wearing a skirt and shes also wearing a suit uh both holding uh suitcases uh they,female the male is wearing a suit and the female is wearing a skirt and shes also wearing a suit both holding suitcases they,0.172,female then the male is wearing uh suit and the female is wearing a skirt and shes also wearing a suit uh both holding uh suitcases uh they,0.069,woman the the man is carryinging and and and woman woman and and and and also also also also also woman also a woman and a woman and suit suites,0.844
2,ebvs_chunk4,EBVS,Spanish,spontaneous,when they when the people are really go go from s from somewhere and they really it doesnt matter because,when the people are really go from somewhere and they really it doesnt matter because,0.250,when the when the people are really go go from so from somewhere and they really it doesnt matter because,0.100,when when the they are uhhhhhhhh uh u u where,0.800
3,bwc_chunk2,BWC,Chinese,spontaneous,and they just falled down and since their cases are the same they didnt pay attention to that so they just grabbed one and go to their go uh go to their destinations,and they just fold it down and since their cases are the same they didnt pay attention to that so they just grab one and go to their destinations,0.235,and they just falled down and since their cases are the same they didnt pay attention to that so they just grab one and go to their go go to their destinations,0.061,and just justied up and they are to their to to their they they they,0.788
4,ebvs_chunk3,EBVS,Spanish,spontaneous,the the real their real suitcase and want and the man look at one red dress and the woman uh yellow tie and it happening,the real sweet case and the men look at one red dress and the woman a yellow tee and it happened in,0.423,the their rare their rare sweet case and one and the man looked at one red dress and the woman a yellow tee and it happened,0.385,there their uh case and the and the man looks at a suitcase and suit suit suit suit suit suit suit suit suit suit suit suit,0.786
5,bwc_chunk3,BWC,Chinese,spontaneous,oh go to their home uh maybe their office but um when the man arrived his office he opened the case and found out that this case belongs to,go to their home and maybe their office but when the man arrived his office he opened the case and found out that this case belongs to,0.103,oh go to their home and maybe their office but when the man arrived his office he opened the case and found out that this case belongs to,0.069,o go to their house and be but the man went to his suit his his suit,0.767
6,ebvs_chunk2,EBVS,Spanish,spontaneous,instead they crash each other and after that ø they get home they figure out like their suitcase like get everyone it not,instead they crash each other and after that when they get home they figure out like their seatquakes like get everyone is not,0.130,instead they crash each other and after that when they get home they figure out like their seat grays like get everyone is not,0.167,instead they they crashed each each they they they they they not not not not,0.750
7,hjk_chunk2,HJK,Korean,spontaneous,the airport he uh ran into a woman and they got just crashed to each other and then since he was in a rush he just,the airport he ran into a wom